# Semantic Mapping with ClimateBERT

Xử lý bước Semantic Mapping từ output của Layout Understanding.
Pretrained ClimateBERT để tạo embedding semantic cho từng đoạn văn.

In [1]:
# Install libraries
!pip install -q transformers torch accelerate sentencepiece tqdm pandas numpy
!pip install -U transformers huggingface_hub

In [2]:
# Import libraries
import os
import json
import re
import unicodedata
from typing import List, Dict, Any, Tuple

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

In [8]:
# Cell 4: Mount Google Drive (Colab only)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted.')
except Exception as e:
    print('Google Drive mount skipped:', e)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted.


In [9]:
# Cell 3: Configuration
INPUT_JSON_PATH = None
OUTPUT_JSON_PATH = None
MODEL_NAME = "climatebert/distilroberta-base-climate-f"
MAX_LENGTH = 256
BATCH_SIZE = 16
ALLOWED_LABELS = {'text', 'figure'}

if INPUT_JSON_PATH is None:
    candidate_paths = [
        '/content/drive/MyDrive/0_layoutlmv3_dataset_normalized.json',
        os.path.join(os.getcwd(), '0_layoutlmv3_dataset_normalized.json'),
    ]
    for path in candidate_paths:
        if os.path.exists(path):
            INPUT_JSON_PATH = path
            break

if OUTPUT_JSON_PATH is None:
    OUTPUT_JSON_PATH = os.path.join(os.getcwd(), 'semantic_mapping_blocks.json')

print('Input JSON:', INPUT_JSON_PATH)
print('Output JSON:', OUTPUT_JSON_PATH)
print('Model:', MODEL_NAME)

Input JSON: /content/drive/MyDrive/0_layoutlmv3_dataset_normalized.json
Output JSON: /content/semantic_mapping_blocks.json
Model: climatebert/distilroberta-base-climate-f


In [10]:
print(MODEL_NAME)


climatebert/distilroberta-base-climate-f


In [11]:
# Cell 5: Load dataset
if INPUT_JSON_PATH is None or not os.path.exists(INPUT_JSON_PATH):
    raise FileNotFoundError('Không tìm thấy file đầu vào. Vui lòng cập nhật INPUT_JSON_PATH.')

with open(INPUT_JSON_PATH, 'r', encoding='utf-8') as f:
    dataset = json.load(f)

print('Số mẫu trong dataset:', len(dataset))
if dataset:
    sample = dataset[0]
    print('Keys:', list(sample.keys()))
    print('Số token đầu tiên:', len(sample.get('words', [])))

Số mẫu trong dataset: 2466
Keys: ['image_id', 'file_name', 'width', 'height', 'words', 'bboxes', 'labels', 'bbox_scale']
Số token đầu tiên: 44


In [12]:
# Cell 6: Filter text & figure blocks
def normalize_text(text: str) -> str:
    text = unicodedata.normalize('NFKC', text)
    text = text.replace(' ', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def build_text_blocks(sample: Dict[str, Any]) -> List[Dict[str, Any]]:
    blocks = []
    current_words = []
    current_boxes = []
    current_labels = []

    for word, box, label in zip(sample.get('words', []), sample.get('bboxes', []), sample.get('labels', [])):
        if label in ALLOWED_LABELS:
            current_words.append(word)
            current_boxes.append(box)
            current_labels.append(label)
        else:
            if current_words:
                blocks.append({
                    'text': normalize_text(' '.join(current_words)),
                    'bbox': [
                        min(b[0] for b in current_boxes),
                        min(b[1] for b in current_boxes),
                        max(b[2] for b in current_boxes),
                        max(b[3] for b in current_boxes),
                    ],
                    'layout_label': current_labels[0],
                })
                current_words = []
                current_boxes = []
                current_labels = []

    if current_words:
        blocks.append({
            'text': normalize_text(' '.join(current_words)),
            'bbox': [
                min(b[0] for b in current_boxes),
                min(b[1] for b in current_boxes),
                max(b[2] for b in current_boxes),
                max(b[3] for b in current_boxes),
            ],
            'layout_label': current_labels[0],
        })

    return blocks

sample_blocks = build_text_blocks(dataset[0])
print('Số block sau khi gom:', len(sample_blocks))
if sample_blocks:
    print('Ví dụ block:', sample_blocks[0])

Số block sau khi gom: 1
Ví dụ block: {'text': "4. SUMMARY OF SIGNIFICANT ACCOUNTING POLICIES (continued) 4.16 Technical reserves (continued) 4.16.1Life insurance reserves (continued) Claim reserve includes provisions for losses that have been notified or claimed but. have not yet been resolved at the date of making technical reserve and provisions for Iosses that have occurred but have not yet been notified or claimed.. Reported but not admitted reserve (RBNA) is calculated for each individual outstanding claim requests and based on the expected sum insured payables for. each case that have been submitted but still in the course of settlement as at the balance sheet date. Reserve for incurred but not reported claims (IBNR) was set aside at 3% of insurance premium collected in fiscal year for periodic payment policy or 3% of single insurance premium divided by insurance term for the single premium payment policy, or the greater of 3% of the premiums used from the beginning of the policy

In [13]:
# Cell 7: Build text blocks for all samples
def extract_page_number(file_name: str) -> int:
    match = re.search(r'[_-]p(\d+)', file_name.lower())
    if match:
        return int(match.group(1))
    return 1


def collect_block_word_bboxes(sample: Dict[str, Any]) -> List[Dict[str, Any]]:
    grouped_blocks = []
    current_words = []
    current_boxes = []
    current_labels = []

    for word, box, label in zip(sample.get('words', []), sample.get('bboxes', []), sample.get('labels', [])):
        if label in ALLOWED_LABELS:
            current_words.append(word)
            current_boxes.append(box)
            current_labels.append(label)
        else:
            if current_words:
                grouped_blocks.append({
                    'text': normalize_text(' '.join(current_words)),
                    'word_bboxes': current_boxes.copy(),
                    'layout_label': current_labels[0],
                })
                current_words = []
                current_boxes = []
                current_labels = []

    if current_words:
        grouped_blocks.append({
            'text': normalize_text(' '.join(current_words)),
            'word_bboxes': current_boxes.copy(),
            'layout_label': current_labels[0],
        })

    return grouped_blocks


all_blocks = []
for sample in tqdm(dataset, desc='Building semantic blocks'):
    blocks = build_text_blocks(sample)
    grouped_blocks = collect_block_word_bboxes(sample)
    if not blocks:
        continue

    page_number = extract_page_number(sample.get('file_name', ''))
    page_width = sample.get('width')
    page_height = sample.get('height')
    file_name = sample.get('file_name', '')

    for block_index, block in enumerate(blocks):
        if not block['text']:
            continue

        grouped_block = grouped_blocks[block_index] if block_index < len(grouped_blocks) else None
        word_bboxes = grouped_block.get('word_bboxes', []) if grouped_block else []

        all_blocks.append({
            'file_name': file_name,
            'page': page_number,
            'layout_label': block['layout_label'],
            'text': block['text'],
            'bbox': block['bbox'],
            'block_bbox': block['bbox'],
            'word_bboxes': word_bboxes,
            'page_width': page_width,
            'page_height': page_height,
            'page_number': page_number,
            'semantic_ready': True,
            'scope_mapping_ready': True,
            'document_type': 'ESG Report',
            'content_type': block['layout_label'],
            'text_length': len(block['text']),
            'language': None,
            'section_id': None,
            'block_index': block_index,
            'parent_page': page_number,
            'metadata': {
                'chunk_id': f'{file_name}__p{page_number}__b{block_index}',
                'page_id': f'{file_name}__p{page_number}',
                'block_index': block_index,
                'parent_page': page_number,
                'page_width': page_width,
                'page_height': page_height,
                'source_file': file_name,
                'source_module': 'LayoutLMv3',
                'semantic_model': MODEL_NAME,
                'embedding_model': MODEL_NAME,
                'embedding_dimension': None,
                'pipeline_stage': 'Semantic Mapping',
                'created_by': 'SemanticMapping',
                'source': 'LayoutLMv3',
                'block_type': block['layout_label'],
            },
        })

print('Tổng số block sau khi gom:', len(all_blocks))

Building semantic blocks:   0%|          | 0/2466 [00:00<?, ?it/s]

Tổng số block sau khi gom: 13188


In [14]:
# Cell 8: Load ClimateBERT pretrained
# Chọn ClimateBERT pretrained phù hợp cho tài liệu ESG vì mô hình này đã học về ngữ cảnh khí hậu, sustainability và carbon-related terminology.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

print('Device:', device)
print('Tokenizer loaded:', MODEL_NAME)
print('Model loaded:', MODEL_NAME)

config.json:   0%|          | 0.00/752 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.46k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.15M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: climatebert/distilroberta-base-climate-f
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Device: cuda
Tokenizer loaded: climatebert/distilroberta-base-climate-f
Model loaded: climatebert/distilroberta-base-climate-f


In [15]:
# Cell 9: Generate embeddings
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    summed = torch.sum(token_embeddings * input_mask_expanded, dim=1)
    counts = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
    return summed / counts

def embed_texts(texts: List[str]) -> List[List[float]]:
    embeddings = []
    for start in tqdm(range(0, len(texts), BATCH_SIZE), desc='Generating embeddings'):
        batch_texts = texts[start:start + BATCH_SIZE]
        encoded = tokenizer(batch_texts, return_tensors='pt', padding=True, truncation=True, max_length=MAX_LENGTH)
        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            outputs = model(**encoded)

        pooled_embeddings = mean_pooling(outputs, encoded['attention_mask'])
        embeddings.extend(pooled_embeddings.cpu().tolist())

    return embeddings

texts = [block['text'] for block in all_blocks]
embeddings = embed_texts(texts)

for block, embedding in zip(all_blocks, embeddings):
    block['embedding'] = embedding
    block['metadata']['embedding_dimension'] = len(embedding)

print('Số embedding được tạo:', len(embeddings))
print('Kích thước embedding đầu tiên:', len(embeddings[0]) if embeddings else 0)

Generating embeddings:   0%|          | 0/825 [00:00<?, ?it/s]

Số embedding được tạo: 13188
Kích thước embedding đầu tiên: 768


In [16]:
# Cell 10: Export JSON
os.makedirs(os.path.dirname(OUTPUT_JSON_PATH) or '.', exist_ok=True)
with open(OUTPUT_JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(all_blocks, f, ensure_ascii=False, indent=2)

print('Đã lưu semantic blocks tại:', OUTPUT_JSON_PATH)
print('Số record xuất ra:', len(all_blocks))
if all_blocks:
    print('Ví dụ record:', all_blocks[0])

Đã lưu semantic blocks tại: /content/semantic_mapping_blocks.json
Số record xuất ra: 13188
Ví dụ record: {'file_name': 'bao_viet_holdings_2024_p345_jpg.rf.a30d88b69fd1694f4aa1e448d0835a10.jpg', 'page': 345, 'layout_label': 'text', 'text': "4. SUMMARY OF SIGNIFICANT ACCOUNTING POLICIES (continued) 4.16 Technical reserves (continued) 4.16.1Life insurance reserves (continued) Claim reserve includes provisions for losses that have been notified or claimed but. have not yet been resolved at the date of making technical reserve and provisions for Iosses that have occurred but have not yet been notified or claimed.. Reported but not admitted reserve (RBNA) is calculated for each individual outstanding claim requests and based on the expected sum insured payables for. each case that have been submitted but still in the course of settlement as at the balance sheet date. Reserve for incurred but not reported claims (IBNR) was set aside at 3% of insurance premium collected in fiscal year for peri

In [17]:
import random
import torch
import torch.nn.functional as F

# -----------------------------
# Mean Pooling
# -----------------------------
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

    return (token_embeddings * mask).sum(1) / torch.clamp(mask.sum(1), min=1e-9)


# -----------------------------
# Encode query
# -----------------------------
def encode_text(text):

    inputs = tokenizer(
        text,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        outputs = model(**inputs)
        emb = mean_pooling(outputs, inputs["attention_mask"])

    emb = F.normalize(emb, p=2, dim=1)

    return emb.squeeze(0).cpu()


# -----------------------------
# Random query
# -----------------------------
sample = random.choice(all_blocks)

query = sample["text"]

print("=" * 80)
print("QUERY")
print("=" * 80)
print(query[:700], "...\n")


# -----------------------------
# Query embedding
# -----------------------------
query_emb = encode_text(query)

scores = []

for item in all_blocks:

    emb = torch.tensor(item["embedding"])

    emb = F.normalize(emb, p=2, dim=0)

    score = F.cosine_similarity(
        query_emb,
        emb,
        dim=0
    ).item()

    scores.append(score)


# -----------------------------
# Top-k
# -----------------------------
topk = torch.topk(torch.tensor(scores), k=5)

print("=" * 80)
print("TOP 5 MOST SIMILAR BLOCKS")
print("=" * 80)

for rank, idx in enumerate(topk.indices):

    rec = all_blocks[idx]

    print(f"\nRank {rank+1}")
    print(f"Similarity : {scores[idx]:.4f}")
    print(f"Page       : {rec['page']}")
    print(f"Label      : {rec['layout_label']}")
    print(f"Text       :")
    print(rec["text"][:400])

QUERY
3.22 Comparative information ...

TOP 5 MOST SIMILAR BLOCKS

Rank 1
Similarity : 1.0000
Page       : 123
Label      : text
Text       :
3.22 Comparative information

Rank 2
Similarity : 0.9906
Page       : 143
Label      : text
Text       :
36. Comparative information

Rank 3
Similarity : 0.9792
Page       : 121
Label      : text
Text       :
3.16 Income tax

Rank 4
Similarity : 0.9791
Page       : 130
Label      : text
Text       :
(v) Comparative information 6.Financial investments

Rank 5
Similarity : 0.9787
Page       : 36
Label      : text
Text       :
2.3. Commercial performance


In [19]:
import torch
import torch.nn.functional as F

def encode_text(text):
    inputs = tokenizer(text, padding=True, truncation=True, max_length=256, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        emb = mean_pooling(outputs, inputs['attention_mask'])
    return F.normalize(emb, p=2, dim=1).squeeze(0)

# Các query mẫu đại diện cho emission-related text
emission_queries = [
    "Scope 1 direct greenhouse gas emissions from fuel combustion",
    "Scope 2 purchased electricity emissions",
    "Scope 3 value chain and supply chain emissions",
    "carbon footprint tCO2e GHG emissions"
]

query_embeddings = [encode_text(q) for q in emission_queries]

def is_emission_candidate(block: dict, sim_threshold: float = 0.68) -> bool:
    text = block.get('text', '')
    if len(text) < 20:
        return False

    text_lower = text.lower()
    keywords = ['scope', 'emission', 'ghg', 'co2', 'tco2', 'carbon', 'fuel', 'electricity',
                'waste', 'combustion', 'purchased', 'generated']

    if any(kw in text_lower for kw in keywords):
        return True

    # Embedding similarity
    if 'embedding' in block:
        block_emb = torch.tensor(block['embedding']).unsqueeze(0).to(device) # Moved to device
        block_emb = F.normalize(block_emb, p=2, dim=1)

        similarities = [F.cosine_similarity(block_emb, q_emb.unsqueeze(0)).item()
                       for q_emb in query_embeddings]
        return max(similarities) >= sim_threshold

    return False


candidates = [block for block in tqdm(all_blocks, desc="Filtering candidates")
              if is_emission_candidate(block)]

print(f"Tổng blocks: {len(all_blocks)} → Candidates: {len(candidates)}")

Filtering candidates:   0%|          | 0/13188 [00:00<?, ?it/s]

Tổng blocks: 13188 → Candidates: 12236


In [20]:
# Cell 12: Baseline Scope Classification (BART-MNLI)
from transformers import pipeline

classifier = pipeline("zero-shot-classification",
                     model="facebook/bart-large-mnli",
                     device=0 if torch.cuda.is_available() else -1)

labels = ["Scope 1", "Scope 2", "Scope 3", "Not emission related"]

def classify_scope(text: str, threshold: float = 0.52):
    result = classifier(text[:512], labels, multi_label=False)
    scope = result['labels'][0]
    conf = result['scores'][0]

    if conf < threshold:
        scope = "Unknown"

    return {
        "term": text[:280] + ("..." if len(text) > 280 else ""),
        "scope": scope,
        "confidence": round(float(conf), 4),
        "all_scores": {k: round(v, 4) for k, v in zip(result['labels'], result['scores'])}
    }

print("Đang chạy baseline classification...")
for block in tqdm(candidates, desc="Classifying Scope"):
    mapping = classify_scope(block['text'])
    block['scope_mapping'] = mapping
    block['has_valid_scope'] = mapping['scope'].startswith("Scope")

print("Hoàn thành!")

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Đang chạy baseline classification...


Classifying Scope:   0%|          | 0/12236 [00:00<?, ?it/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Hoàn thành!


In [21]:
# Cell 13: Export Final Semantic Mapping Result
FINAL_OUTPUT_PATH = os.path.join(os.getcwd(), 'semantic_mapping_with_scope.json')

# Lưu tất cả blocks (để giữ đầy đủ) và riêng file candidates
with open(FINAL_OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(all_blocks, f, ensure_ascii=False, indent=2)

# Lưu riêng file candidates (dễ dùng sau)
CANDIDATES_PATH = os.path.join(os.getcwd(), 'semantic_candidates_with_scope.json')
with open(CANDIDATES_PATH, 'w', encoding='utf-8') as f:
    json.dump(candidates, f, ensure_ascii=False, indent=2)

print(f"Đã lưu full semantic mapping tại: {FINAL_OUTPUT_PATH}")
print(f"Đã lưu candidates với Scope tại: {CANDIDATES_PATH}")
print(f"Tổng số record: {len(all_blocks)} | Candidates: {len(candidates)}")

Đã lưu full semantic mapping tại: /content/semantic_mapping_with_scope.json
Đã lưu candidates với Scope tại: /content/semantic_candidates_with_scope.json
Tổng số record: 13188 | Candidates: 12236


In [22]:
# Cell 14: Statistics
from collections import Counter

scopes = [b['scope_mapping']['scope'] for b in candidates if 'scope_mapping' in b]
scope_counter = Counter(scopes)

print("Phân bố Scope:")
for scope, count in scope_counter.most_common():
    print(f"  {scope}: {count} blocks")

Phân bố Scope:
  Unknown: 11559 blocks
  Scope 2: 257 blocks
  Scope 1: 204 blocks
  Scope 3: 189 blocks
  Not emission related: 27 blocks


In [23]:
import json
from pprint import pprint

# Đường dẫn đến file bạn muốn xem
JSON_FILE = "/content/semantic_candidates_with_scope.json"

with open(JSON_FILE, 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"Tổng số mẫu trong file: {len(data)}\n")

# Lấy vài mẫu có chất lượng tốt
def show_samples(n=5, min_confidence=0.7):
    # Ưu tiên các mẫu có confidence cao và scope rõ ràng
    good_samples = [item for item in data
                   if item.get('scope_mapping')
                   and item['scope_mapping'].get('confidence', 0) >= min_confidence
                   and item['scope_mapping'].get('scope') != "Unknown"]

    samples = good_samples[:n] if good_samples else data[:n]

    for i, sample in enumerate(samples, 1):
        mapping = sample.get('scope_mapping', {})

        print(f"\n{'='*90}")
        print(f"MẪU {i} | Page {sample.get('page')} | File: {sample.get('file_name')}")
        print(f"{'='*90}")
        print(f"Text ({len(sample['text'])} ký tự):")
        print(sample['text'][:450] + "..." if len(sample['text']) > 450 else sample['text'])
        print("\nKết quả phân loại:")
        print(f"   Term       : {mapping.get('term', 'N/A')[:200]}")
        print(f"   Scope      : {mapping.get('scope', 'N/A')}")
        print(f"   Confidence : {mapping.get('confidence', 'N/A')}")
        print(f"   Block type : {sample.get('layout_label')}")
        print(f"   Bbox       : {sample.get('bbox')}")

# Chạy demo
show_samples(n=10)

Tổng số mẫu trong file: 12236


MẪU 1 | Page 76 | File: hoa_phat_group_jsc_2024_p076_jpg.rf.d8d668fcb9d5596cef385d05975c4536.jpg
Text (77 ký tự):
Phase 2 of the E-office project - document digitization and the establishment

Kết quả phân loại:
   Term       : Phase 2 of the E-office project - document digitization and the establishment
   Scope      : Scope 2
   Confidence : 0.885
   Block type : text
   Bbox       : [196, 564, 547, 590]

MẪU 2 | Page 5 | File: hoa_phat_group_jsc_2025_p005_jpg.rf.b9844b3c036b44568633a130bcc44973.jpg
Text (1159 ký tự):
REPORTING FRAMEWORK [GRI 1] The Hoa Phat Group Sustainability Report 2025 has been prepared in accordance with the Global Reporting. Initiative (GRl) Sustainability Reporting Standards 2021, for the period from 1 January 2025 to 31 December. 2025, with the disclosed information referenced in the GRI content index table. The report also reflect the. Sustainability Accounting Standards Board (SAsB) standards for iron and steel producers. Th